# Feature Engineering

---

1. Import packages
2. Load data
3. Feature engineering

---

## 1. Import packages

In [1]:
import pandas as pd

---
## 2. Load data

In [2]:
df = pd.read_csv('./clean_data_after_eda.csv')
df["date_activ"] = pd.to_datetime(df["date_activ"], format='%Y-%m-%d')
df["date_end"] = pd.to_datetime(df["date_end"], format='%Y-%m-%d')
df["date_modif_prod"] = pd.to_datetime(df["date_modif_prod"], format='%Y-%m-%d')
df["date_renewal"] = pd.to_datetime(df["date_renewal"], format='%Y-%m-%d')

In [12]:
df.shape

(14606, 66)

In [3]:
df.head(3)

,id,channel_sales,cons_12m,cons_gas_12m,cons_last_month,date_activ,date_end,date_modif_prod,date_renewal,forecast_cons_12m,...,var_6m_price_off_peak_var,var_6m_price_peak_var,var_6m_price_mid_peak_var,var_6m_price_off_peak_fix,var_6m_price_peak_fix,var_6m_price_mid_peak_fix,var_6m_price_off_peak,var_6m_price_peak,var_6m_price_mid_peak,churn
0,24011ae4ebbe3035111d65fa7c15bc57,foosdfpfkusacimwkcsosbicdxkicaua,0,54946,0,2013-06-15,2016-06-15,2015-11-01,2015-06-23,0.00,...,0.000131,4.100838e-05,0.000908,2.086294,99.530517,44.235794,2.086425,9.953056e+01,44.236702,1
1,d29c2c54acc38ff3c0614d0a653813dd,MISSING,4660,0,0,2009-08-21,2016-08-30,2009-08-21,2015-08-31,189.95,...,0.000003,1.217891e-03,0.000000,0.009482,0.000000,0.000000,0.009485,1.217891e-03,0.000000,0
2,764c75f661154dac3a6c254cd082ea7d,foosdfpfkusacimwkcsosbicdxkicaua,544,0,0,2010-04-16,2016-04-16,2010-04-16,2015-04-17,47.96,...,0.000004,9.450150e-08,0.000000,0.000000,0.000000,0.000000,0.000004,9.450150e-08,0.000000,0


---

## 3. Feature engineering

### Difference between off-peak prices in December and preceding January

Below is the code created by your colleague to calculate the feature described above. Use this code to re-create this feature and then think about ways to build on this feature to create features with a higher predictive power.

In [4]:
price_df = pd.read_csv('../data-analysis/price_data.csv')
price_df["price_date"] = pd.to_datetime(price_df["price_date"], format='%Y-%m-%d')
price_df.head()

,id,price_date,price_off_peak_var,price_peak_var,price_mid_peak_var,price_off_peak_fix,price_peak_fix,price_mid_peak_fix
0,038af19179925da21a25619c5a24b745,2015-01-01,0.151367,0.0,0.0,44.266931,0.0,0.0
1,038af19179925da21a25619c5a24b745,2015-02-01,0.151367,0.0,0.0,44.266931,0.0,0.0
2,038af19179925da21a25619c5a24b745,2015-03-01,0.151367,0.0,0.0,44.266931,0.0,0.0
3,038af19179925da21a25619c5a24b745,2015-04-01,0.149626,0.0,0.0,44.266931,0.0,0.0
4,038af19179925da21a25619c5a24b745,2015-05-01,0.149626,0.0,0.0,44.266931,0.0,0.0


In [5]:
# Group off-peak prices by companies and month
monthly_price_by_id = price_df.groupby(['id', 'price_date']).agg({'price_off_peak_var': 'mean', 'price_off_peak_fix': 'mean'}).reset_index()

# Get january and december prices
jan_prices = monthly_price_by_id.groupby('id').first().reset_index()
dec_prices = monthly_price_by_id.groupby('id').last().reset_index()

# Calculate the difference
diff = pd.merge(dec_prices.rename(columns={'price_off_peak_var': 'dec_1', 'price_off_peak_fix': 'dec_2'}), jan_prices.drop(columns='price_date'), on='id')
diff['offpeak_diff_dec_january_energy'] = diff['dec_1'] - diff['price_off_peak_var']
diff['offpeak_diff_dec_january_power'] = diff['dec_2'] - diff['price_off_peak_fix']
diff = diff[['id', 'offpeak_diff_dec_january_energy','offpeak_diff_dec_january_power']]
diff.head()

,id,offpeak_diff_dec_january_energy,offpeak_diff_dec_january_power
0,0002203ffbb812588b632b9e628cc38d,-0.006192,0.162916
1,0004351ebdd665e6ee664792efc4fd13,-0.004104,0.177779
2,0010bcc39e42b3c2131ed2ce55246e3c,0.050443,1.500000
3,0010ee3855fdea87602a5b7aba8e42de,-0.010018,0.162916
4,00114d74e963e47177db89bc70108537,-0.003994,-0.000001


Now it is time to get creative and to conduct some of your own feature engineering! Have fun with it, explore different ideas and try to create as many as you can!

In [6]:
# attach collegue's code to merge the difference with the main dataframe

df = df.merge(diff, how='left', on='id')

df.head(5)

,id,channel_sales,cons_12m,cons_gas_12m,cons_last_month,date_activ,date_end,date_modif_prod,date_renewal,forecast_cons_12m,...,var_6m_price_mid_peak_var,var_6m_price_off_peak_fix,var_6m_price_peak_fix,var_6m_price_mid_peak_fix,var_6m_price_off_peak,var_6m_price_peak,var_6m_price_mid_peak,churn,offpeak_diff_dec_january_energy,offpeak_diff_dec_january_power
0,24011ae4ebbe3035111d65fa7c15bc57,foosdfpfkusacimwkcsosbicdxkicaua,0,54946,0,2013-06-15,2016-06-15,2015-11-01,2015-06-23,0.00,...,9.084737e-04,2.086294,99.530517,44.235794,2.086425,9.953056e+01,4.423670e+01,1,0.020057,3.700961
1,d29c2c54acc38ff3c0614d0a653813dd,MISSING,4660,0,0,2009-08-21,2016-08-30,2009-08-21,2015-08-31,189.95,...,0.000000e+00,0.009482,0.000000,0.000000,0.009485,1.217891e-03,0.000000e+00,0,-0.003767,0.177779
2,764c75f661154dac3a6c254cd082ea7d,foosdfpfkusacimwkcsosbicdxkicaua,544,0,0,2010-04-16,2016-04-16,2010-04-16,2015-04-17,47.96,...,0.000000e+00,0.000000,0.000000,0.000000,0.000004,9.450150e-08,0.000000e+00,0,-0.004670,0.177779
3,bba03439a292a1e166f80264c16191cb,lmkebamcaaclubfxadlmueccxoimlema,1584,0,0,2010-03-30,2016-03-30,2010-03-30,2015-03-31,240.04,...,0.000000e+00,0.000000,0.000000,0.000000,0.000003,0.000000e+00,0.000000e+00,0,-0.004547,0.177779
4,149d57cf92fc41cf94415803a877cb4b,MISSING,4425,0,526,2010-01-13,2016-03-07,2010-01-13,2015-03-09,445.75,...,4.860000e-10,0.000000,0.000000,0.000000,0.000011,2.896760e-06,4.860000e-10,0,-0.006192,0.162916


In [7]:
# dec minus jan prices for all six prices 

price_cols = ['price_off_peak_var', 'price_off_peak_fix', 'price_peak_var', 'price_peak_fix', 'price_mid_peak_var', 'price_mid_peak_fix']

price_sorted = price_df.sort_values(by=['id', 'price_date'])

g = price_sorted.groupby('id')[price_cols]

dec_jan = (g.last() - g.first()).add_prefix('diff_dec_jan_').reset_index()

# g.head()

dec_jan

,id,diff_dec_jan_price_off_peak_var,diff_dec_jan_price_off_peak_fix,diff_dec_jan_price_peak_var,diff_dec_jan_price_peak_fix,diff_dec_jan_price_mid_peak_var,diff_dec_jan_price_mid_peak_fix
0,0002203ffbb812588b632b9e628cc38d,-0.006192,0.162916,-0.002302,0.097749,0.003487,0.065166
1,0004351ebdd665e6ee664792efc4fd13,-0.004104,0.177779,0.000000,0.000000,0.000000,0.000000
2,0010bcc39e42b3c2131ed2ce55246e3c,0.050443,1.500000,0.000000,0.000000,0.000000,0.000000
3,0010ee3855fdea87602a5b7aba8e42de,-0.010018,0.162916,-0.005120,0.097749,0.000763,0.065166
4,00114d74e963e47177db89bc70108537,-0.003994,-0.000001,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...
16091,ffef185810e44254c3a4c6395e6b4d8a,-0.050232,-0.335085,-0.038788,-0.400251,-0.022735,-0.432834
16092,fffac626da707b1b5ab11e8431a4d0a2,-0.003778,0.177779,0.000000,0.000000,0.000000,0.000000
16093,fffc0cacd305dd51f316424bbb08d1bd,-0.001760,0.164916,-0.003707,0.099749,-0.007326,0.067166
16094,fffe4f5646aa39c7f97f95ae2679ce64,-0.009391,0.162916,-0.004937,0.097749,0.001029,0.065166


In [8]:
# mean and std for each price per customer

stats = g.agg(['mean', 'std'])
stats.columns = [f'{col}_{stat}' for col, stat in stats.columns]

stats = stats.reset_index()

stats.head()

,id,price_off_peak_var_mean,price_off_peak_var_std,price_off_peak_fix_mean,price_off_peak_fix_std,price_peak_var_mean,price_peak_var_std,price_peak_fix_mean,price_peak_fix_std,price_mid_peak_var_mean,price_mid_peak_var_std,price_mid_peak_fix_mean,price_mid_peak_fix_std
0,0002203ffbb812588b632b9e628cc38d,0.124338,0.003976,40.701732,6.341481e-02,0.103794,0.001989,24.421038,0.038049,0.073160,0.001368,16.280694,0.025366
1,0004351ebdd665e6ee664792efc4fd13,0.146426,0.002197,44.385450,8.753223e-02,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,0010bcc39e42b3c2131ed2ce55246e3c,0.181558,0.026008,45.319710,7.723930e-01,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,0010ee3855fdea87602a5b7aba8e42de,0.118757,0.005049,40.647427,8.507958e-02,0.098292,0.002580,24.388455,0.051048,0.069032,0.000403,16.258971,0.034032
4,00114d74e963e47177db89bc70108537,0.147926,0.002202,44.266930,5.908392e-07,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [9]:
# largest month-to-month rise (off-peak energy and power)

month_to_month = price_sorted.copy()

for col in ['price_off_peak_var', 'price_off_peak_fix']:
    month_to_month[f'{col}_mom'] = month_to_month.groupby('id')[col].diff()

# month_to_month.head()

max_jump = month_to_month.groupby('id')[['price_off_peak_var_mom', 'price_off_peak_fix_mom']].max().rename(columns=lambda c: 'max_jump_' + c.replace('_mom', '')).reset_index()


max_jump

,id,max_jump_price_off_peak_var,max_jump_price_off_peak_fix
0,0002203ffbb812588b632b9e628cc38d,0.001969,0.162916
1,0004351ebdd665e6ee664792efc4fd13,0.000358,0.177779
2,0010bcc39e42b3c2131ed2ce55246e3c,0.052694,1.500000
3,0010ee3855fdea87602a5b7aba8e42de,0.000000,0.162912
4,00114d74e963e47177db89bc70108537,0.000468,0.000000
...,...,...,...
16091,ffef185810e44254c3a4c6395e6b4d8a,0.002317,0.000000
16092,fffac626da707b1b5ab11e8431a4d0a2,0.000684,0.177780
16093,fffc0cacd305dd51f316424bbb08d1bd,0.004084,0.164916
16094,fffe4f5646aa39c7f97f95ae2679ce64,0.000000,0.162912


In [10]:
# join to df and check for missing values

print("Missing values in the final dataset:")
print(df.isnull().sum())

for feats in (dec_jan, stats, max_jump):
    df = df.merge(feats, how='left', on='id')

print("\n\nMissing values in the final dataset after merging new features:")
new_cols = [c for c in df.columns if c.startswith(('diff_dec_jan_', 'max_jump_')) or c.endswith(('_mean', '_std'))]
df[new_cols].isna().sum()

Missing values in the final dataset:
id                                 0
channel_sales                      0
cons_12m                           0
cons_gas_12m                       0
cons_last_month                    0
date_activ                         0
date_end                           0
date_modif_prod                    0
date_renewal                       0
forecast_cons_12m                  0
forecast_cons_year                 0
forecast_discount_energy           0
forecast_meter_rent_12m            0
forecast_price_energy_off_peak     0
forecast_price_energy_peak         0
forecast_price_pow_off_peak        0
has_gas                            0
imp_cons                           0
margin_gross_pow_ele               0
margin_net_pow_ele                 0
nb_prod_act                        0
net_margin                         0
num_years_antig                    0
origin_up                          0
pow_max                            0
var_year_price_off_peak_var        0
v

diff_dec_jan_price_off_peak_var    0
diff_dec_jan_price_off_peak_fix    0
diff_dec_jan_price_peak_var        0
diff_dec_jan_price_peak_fix        0
diff_dec_jan_price_mid_peak_var    0
diff_dec_jan_price_mid_peak_fix    0
price_off_peak_var_mean            0
price_off_peak_var_std             0
price_off_peak_fix_mean            0
price_off_peak_fix_std             0
price_peak_var_mean                0
price_peak_var_std                 0
price_peak_fix_mean                0
price_peak_fix_std                 0
price_mid_peak_var_mean            0
price_mid_peak_var_std             0
price_mid_peak_fix_mean            0
price_mid_peak_fix_std             0
max_jump_price_off_peak_var        0
max_jump_price_off_peak_fix        0
dtype: int64

In [14]:
df.groupby('churn')[new_cols].mean().T

churn,0,1
diff_dec_jan_price_off_peak_var,-0.004561,-0.004605
diff_dec_jan_price_off_peak_fix,0.277313,0.289483
diff_dec_jan_price_peak_var,-0.000812,-0.000960
diff_dec_jan_price_peak_fix,-0.042336,-0.080595
diff_dec_jan_price_mid_peak_var,0.000480,0.000484
diff_dec_jan_price_mid_peak_fix,-0.024576,-0.037461
price_off_peak_var_mean,0.142375,0.141887
price_off_peak_var_std,0.004008,0.004635
price_off_peak_fix_mean,42.903812,43.161941
price_off_peak_fix_std,0.182317,0.247066


In [15]:
df[new_cols + ['churn']].corr()['churn'].drop('churn').sort_values()

price_off_peak_var_mean           -0.006421
diff_dec_jan_price_peak_fix       -0.005183
diff_dec_jan_price_peak_var       -0.004041
diff_dec_jan_price_mid_peak_fix   -0.002681
diff_dec_jan_price_off_peak_var   -0.001064
diff_dec_jan_price_mid_peak_var    0.000191
diff_dec_jan_price_off_peak_fix    0.002672
price_mid_peak_fix_std             0.012263
price_peak_fix_std                 0.015224
price_peak_var_std                 0.016153
price_off_peak_fix_mean            0.016800
max_jump_price_off_peak_fix        0.019442
max_jump_price_off_peak_var        0.021377
price_mid_peak_var_std             0.021828
price_off_peak_fix_std             0.023713
price_peak_var_mean                0.029588
price_off_peak_var_std             0.037397
price_mid_peak_fix_mean            0.044842
price_mid_peak_var_mean            0.046544
price_peak_fix_mean                0.047240
Name: churn, dtype: float64

In [16]:
df[new_cols].corr().abs().round(2)

,diff_dec_jan_price_off_peak_var,diff_dec_jan_price_off_peak_fix,diff_dec_jan_price_peak_var,diff_dec_jan_price_peak_fix,diff_dec_jan_price_mid_peak_var,diff_dec_jan_price_mid_peak_fix,price_off_peak_var_mean,price_off_peak_var_std,price_off_peak_fix_mean,price_off_peak_fix_std,price_peak_var_mean,price_peak_var_std,price_peak_fix_mean,price_peak_fix_std,price_mid_peak_var_mean,price_mid_peak_var_std,price_mid_peak_fix_mean,price_mid_peak_fix_std,max_jump_price_off_peak_var,max_jump_price_off_peak_fix
diff_dec_jan_price_off_peak_var,1.00,0.47,0.18,0.18,0.06,0.17,0.28,0.41,0.07,0.09,0.11,0.12,0.18,0.03,0.17,0.07,0.18,0.03,0.69,0.17
diff_dec_jan_price_off_peak_fix,0.47,1.00,0.04,0.01,0.05,0.04,0.17,0.13,0.10,0.02,0.05,0.02,0.08,0.01,0.07,0.01,0.07,0.01,0.28,0.29
diff_dec_jan_price_peak_var,0.18,0.04,1.00,0.50,0.57,0.51,0.17,0.07,0.07,0.06,0.11,0.10,0.18,0.14,0.17,0.10,0.18,0.14,0.12,0.04
diff_dec_jan_price_peak_fix,0.18,0.01,0.50,1.00,0.90,0.98,0.00,0.11,0.01,0.16,0.01,0.13,0.01,0.27,0.00,0.25,0.00,0.28,0.17,0.11
diff_dec_jan_price_mid_peak_var,0.06,0.05,0.57,0.90,1.00,0.90,0.06,0.03,0.03,0.13,0.09,0.06,0.11,0.22,0.12,0.13,0.11,0.23,0.06,0.10
diff_dec_jan_price_mid_peak_fix,0.17,0.04,0.51,0.98,0.90,1.00,0.01,0.10,0.01,0.15,0.00,0.13,0.01,0.27,0.01,0.26,0.01,0.28,0.16,0.11
price_off_peak_var_mean,0.28,0.17,0.17,0.00,0.06,0.01,1.00,0.11,0.66,0.04,0.32,0.01,0.63,0.10,0.61,0.15,0.61,0.10,0.20,0.03
price_off_peak_var_std,0.41,0.13,0.07,0.11,0.03,0.10,0.11,1.00,0.02,0.58,0.18,0.33,0.17,0.32,0.18,0.38,0.17,0.32,0.85,0.51
price_off_peak_fix_mean,0.07,0.10,0.07,0.01,0.03,0.01,0.66,0.02,1.00,0.01,0.21,0.01,0.23,0.04,0.29,0.09,0.33,0.07,0.02,0.02
price_off_peak_fix_std,0.09,0.02,0.06,0.16,0.13,0.15,0.04,0.58,0.01,1.00,0.02,0.31,0.02,0.50,0.02,0.45,0.01,0.47,0.55,0.91
